# Step 2 — Score 14 Constitutional Dimensions

Loads `typology/ccpc_typology_v4.json` (from Step 0) and applies it to all 17k country-years
in CCPCNC v5, producing a 0–1 score for each of the 14 dimensions.

**Key design:**
- Missing/inapplicable codes (96, 97, 98, 99) → `NaN`, excluded from the weighted average
- Explicit 'No' answers (e.g. code 2 → 0.0) are included — absence of a right is signal
- Denominator = sum of weights for variables with *valid* scores on that row
  (prevents zero-inflation from conditional sub-variables)

**Input:**  `ccpc_typology.json`  
**Output:** `ccpc_axis_scores_llm.csv`  (read by Step 3)

In [1]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


In [2]:
TYPOLOGY_PATH  = 'typology/ccpc_typology_v4.json'
FULL_DATA_PATH = 'data/ccpcnc/ccpcnc_v5.csv'
SCORES_PATH    = 'ccpc_axis_scores_llm.csv'

DIMENSIONS = [
    'civil_liberties',
    'socioeconomic_rights',
    'political_competition',
    'legislative_autonomy',
    'executive_constraints',
    'judicial_independence',
    'rule_of_law_due_process',
    'institutional_accountability',
    'emergency_powers_constraints',
    'civilian_control_of_security',
    'amendment_rigidity',
    'federalism_decentralization',
    'transparency_information_access',
    'equality_gender_minority_indigenous',
]

# CCPCNC codes that mean missing / inapplicable — excluded from scoring
MISSING_CODES = {90.0, 96.0, 97.0, 98.0, 99.0}

print(f'Typology:  {TYPOLOGY_PATH}')
print(f'Dataset:   {FULL_DATA_PATH}')
print(f'Output:    {SCORES_PATH}')

Typology:  typology/ccpc_typology_v4.json
Dataset:   data/ccpcnc/ccpcnc_v5.csv
Output:    ccpc_axis_scores_llm.csv


## 1 — Load Typology

In [3]:
with open(TYPOLOGY_PATH) as f:
    typology = json.load(f)

from collections import Counter
counts = Counter(dim for info in typology.values() for dim in info.get('dimensions', {}))

print(f'Loaded: {len(typology)} variables')
print('\nVariables per dimension:')
for dim in DIMENSIONS:
    print(f'  {dim:45s} {counts.get(dim, 0):3d}')

Loaded: 839 variables

Variables per dimension:
  civil_liberties                               114
  socioeconomic_rights                           79
  political_competition                         161
  legislative_autonomy                          149
  executive_constraints                         133
  judicial_independence                         246
  rule_of_law_due_process                       127
  institutional_accountability                   49
  emergency_powers_constraints                   30
  civilian_control_of_security                   23
  amendment_rigidity                             27
  federalism_decentralization                    44
  transparency_information_access                22
  equality_gender_minority_indigenous           150


## 2 — Load CCPC Dataset

In [4]:
ccpc_raw = pd.read_csv(FULL_DATA_PATH, low_memory=False)
d = ccpc_raw[
    (ccpc_raw['year'] <= 2023) &
    ccpc_raw['systyear'].notna()
].sort_values(['cowcode', 'year']).copy()

print(f'Rows:      {len(d):,}')
print(f'Countries: {d["cowcode"].nunique()}')
print(f'Years:     {d["year"].min():.0f}–{d["year"].max():.0f}')

Rows:      17,390
Countries: 212
Years:     1789–2023


## 3 — Score Each Variable

In [5]:
def apply_value_map(series: pd.Series, value_map: dict) -> pd.Series:
    """
    Map raw CCPCNC codes → [0, 1] scores.

    - Missing/inapplicable codes (90/96/97/98/99) → NaN (excluded from denominator)
    - Explicit 'No' (e.g. code 2 → 0.0)          → 0.0 (included; absence is signal)
    - Raw NaN                                      → NaN
    - __numeric__ marker: normalize by 5th–95th percentile
    """
    if value_map.get('1') == '__numeric__' or value_map.get('__numeric__'):
        s = pd.to_numeric(series, errors='coerce')
        s[s.isin(MISSING_CODES)] = np.nan
        lo, hi = s.quantile(0.05), s.quantile(0.95)
        if hi > lo:
            return (s.clip(lo, hi) - lo) / (hi - lo)
        return pd.Series(np.nan, index=series.index)

    result = pd.Series(np.nan, index=series.index, dtype=float)
    numeric = pd.to_numeric(series, errors='coerce')

    for code_str, score in value_map.items():
        try:
            code = float(code_str)
        except (ValueError, TypeError):
            continue
        if code in MISSING_CODES:
            continue
        result[numeric == code] = float(score)

    return result


scored = {}
missing_vars = []

for var, info in typology.items():
    if var not in d.columns:
        missing_vars.append(var)
        continue
    scored[var] = apply_value_map(d[var], info['value_map'])

print(f'Scored:           {len(scored)} variables')
print(f'Not in dataset:   {len(missing_vars)} variables')

Scored:           835 variables
Not in dataset:   4 variables


## 4 — Aggregate to 14 Dimension Scores

In [6]:
def compute_dimension_score(dim: str) -> np.ndarray:
    """
    Weighted average over variables in this dimension.
    Denominator = sum of weights for rows where the variable has a valid score.
    Rows with zero valid variables → NaN.
    """
    dim_vars = {
        var: info['dimensions'][dim]
        for var, info in typology.items()
        if dim in info.get('dimensions', {}) and var in scored
    }
    if not dim_vars:
        return np.full(len(d), np.nan)

    cols    = list(dim_vars.keys())
    weights = np.array([dim_vars[c] for c in cols], dtype=float)
    vals    = np.column_stack([scored[c].values for c in cols])

    valid            = ~np.isnan(vals)
    effective_weight = (valid * weights).sum(axis=1)
    weighted_sum     = np.where(valid, vals * weights, 0.0).sum(axis=1)

    score = np.where(effective_weight > 0, weighted_sum / effective_weight, np.nan)
    return np.clip(score, 0.0, 1.0)


score_cols = []
for dim in DIMENSIONS:
    col = f'ccpc_{dim}'
    d[col] = compute_dimension_score(dim)
    score_cols.append(col)

print('Dimension score distributions:')
print(d[score_cols].describe().round(3))

Dimension score distributions:
       ccpc_civil_liberties  ccpc_socioeconomic_rights  \
count             14757.000                  14757.000   
mean                  0.469                      0.282   
std                   0.144                      0.204   
min                   0.107                      0.016   
25%                   0.395                      0.103   
50%                   0.468                      0.234   
75%                   0.566                      0.452   
max                   0.815                      0.836   

       ccpc_political_competition  ccpc_legislative_autonomy  \
count                   14757.000                  14757.000   
mean                        0.621                      0.516   
std                         0.082                      0.055   
min                         0.312                      0.179   
25%                         0.559                      0.486   
50%                         0.632                      0.519  

## 5 — Diagnostics

In [7]:
print(f'{"Dimension":50s}  {"Mean":>6}  {"NaN%":>6}  {"Zero%":>6}')
print('-' * 75)
for col in score_cols:
    mean   = d[col].mean()
    pct_nan  = d[col].isna().mean() * 100
    pct_zero = (d[col] == 0).mean() * 100
    print(f'{col:50s}  {mean:6.3f}  {pct_nan:5.1f}%  {pct_zero:5.1f}%')

Dimension                                             Mean    NaN%   Zero%
---------------------------------------------------------------------------
ccpc_civil_liberties                                 0.469   15.1%    0.0%
ccpc_socioeconomic_rights                            0.282   15.1%    0.0%
ccpc_political_competition                           0.621   15.1%    0.0%
ccpc_legislative_autonomy                            0.516   15.1%    0.0%
ccpc_executive_constraints                           0.422   15.1%    0.0%
ccpc_judicial_independence                           0.492   15.1%    0.0%
ccpc_rule_of_law_due_process                         0.414   15.1%    0.0%
ccpc_institutional_accountability                    0.360   15.1%    0.1%
ccpc_emergency_powers_constraints                    0.426   15.1%    0.0%
ccpc_civilian_control_of_security                    0.453   15.1%    0.0%
ccpc_amendment_rigidity                              0.611   15.3%    4.0%
ccpc_federalism_decentra

## 6 — Export

In [8]:
output_cols = ['cowcode', 'country', 'systyear', 'year'] + score_cols
ccpc_scores = d[output_cols].astype({'year': int}).copy()
ccpc_scores.to_csv(SCORES_PATH, index=False)

print(f'Exported: {SCORES_PATH}')
print(f'  Rows:      {len(ccpc_scores):,}')
print(f'  Countries: {ccpc_scores["cowcode"].nunique()}')
print(f'  Years:     {ccpc_scores["year"].min()}–{ccpc_scores["year"].max()}')
print('\nReady for Step 3.')

Exported: ccpc_axis_scores_llm.csv
  Rows:      17,390
  Countries: 212
  Years:     1789–2023

Ready for Step 3.
